# 🔌 bitcoin.API.ipynb
This notebook demonstrates how to:
- Fetch live Bitcoin data from the [CoinGecko API](https://www.coingecko.com/en/api)
- Format it to match the Protobuf schema (`bitcoin_full.proto`)
- Serialize it using `bitcoin_full_pb2.py`
- Save to disk and read it back

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), 'src'))

import requests
import time
from bitcoin_full_pb2 import BitcoinFullData

## 📡 Step 1: Fetch Bitcoin Data from CoinGecko

In [ ]:
url = "https://api.coingecko.com/api/v3/coins/markets"
params = {"vs_currency": "usd", "ids": "bitcoin"}
response = requests.get(url, params=params)
btc = response.json()[0]  # Extract first (and only) item
btc

## 🛠 Step 2: Format Data for Protobuf

In [ ]:
btc_dict = {
    "timestamp": int(time.time()),
    "id": btc["id"],
    "symbol": btc["symbol"],
    "name": btc["name"],
    "image": btc["image"],
    "current_price": btc["current_price"],
    "market_cap": btc["market_cap"],
    "market_cap_rank": btc["market_cap_rank"],
    "fully_diluted_valuation": btc.get("fully_diluted_valuation", 0.0),
    "total_volume": btc["total_volume"],
    "high_24h": btc["high_24h"],
    "low_24h": btc["low_24h"],
    "price_change_24h": btc["price_change_24h"],
    "price_change_percentage_24h": btc["price_change_percentage_24h"],
    "market_cap_change_24h": btc["market_cap_change_24h"],
    "market_cap_change_percentage_24h": btc["market_cap_change_percentage_24h"],
    "circulating_supply": btc["circulating_supply"],
    "total_supply": btc.get("total_supply", 0.0),
    "max_supply": btc.get("max_supply", 0.0),
    "ath": btc["ath"],
    "ath_change_percentage": btc["ath_change_percentage"],
    "ath_date": btc["ath_date"],
    "atl": btc["atl"],
    "atl_change_percentage": btc["atl_change_percentage"],
    "atl_date": btc["atl_date"],
    "last_updated": btc["last_updated"],
    "source": "CoinGecko"
}

btc_dict

## 📦 Step 3: Serialize to Protobuf Format

In [ ]:
msg = BitcoinFullData(**btc_dict)
serialized = msg.SerializeToString()
print(f"✅ Serialized {len(serialized)} bytes")

## 💾 Step 4: Save to Disk and Read Back

In [ ]:
file_path = "src/data/sample_single_record.pb"
with open(file_path, "wb") as f:
    f.write(len(serialized).to_bytes(4, 'little'))
    f.write(serialized)

# Read back
with open(file_path, "rb") as f:
    length_bytes = f.read(4)
    size = int.from_bytes(length_bytes, 'little')
    message_data = f.read(size)

msg_read = BitcoinFullData()
msg_read.ParseFromString(message_data)
msg_read